# Explore Rebirth history with SQL

Run the cells from top to bottom. DuckDB reads the governed Parquet archive directly in memory: there is no server, login, or database file to manage.

In [ ]:
import importlib
from pathlib import Path
import sys

project_root = next(
    path.resolve()
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "core").is_dir() and (path / "data" / "histo").is_dir()
)
sys.path.insert(0, str(project_root))

open_history_database = importlib.import_module(
    "core.archive_sql"
).open_history_database
db = open_history_database(project_root / "data" / "histo")

The useful views are `archive_days`, `risk_history`, `market_history`, `colossus_history`, and `stock_history`.

In [ ]:
db.sql("""
    SELECT count(*) AS days,
           min("Snapshot Date") AS first_date,
           max("Snapshot Date") AS last_date
    FROM archive_days
""").df()

In [ ]:
db.sql("""
    SELECT "Snapshot Date", round(sum("Risk"), 2) AS total_risk
    FROM risk_history
    WHERE "Risk Type" = 'FX' AND "Risk Greek" = 'Delta'
    GROUP BY "Snapshot Date"
    ORDER BY "Snapshot Date" DESC
    LIMIT 20
""").df()